## BONUS: RabbitMQ to FreshCart Snowflake Pipeline

### Step 1: Verify the Python Environment and Existing Pipeline Functions

This cell checks whether the RabbitMQ library, Snowflake connection,
and FreshCart pipeline functions are available before starting the
bonus implementation.

## BONUS — Step 1.1: Install RabbitMQ Dependencies

Install the RabbitMQ Python client in the currently selected
Jupyter notebook environment.

In [2]:
# BONUS — STEP 1.1: INSTALL RABBITMQ DEPENDENCIES

%pip install pika python-dotenv

  Using cached pika-1.4.4-py3-none-any.whl.metadata (13 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached pika-1.4.4-py3-none-any.whl (165 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)

   -------------------- ------------------- 1/2 [pika]
   -------------------- ------------------- 1/2 [pika]
   -------------------- ------------------- 1/2 [pika]
   ---------------------------------------- 2/2 [pika]

Note: you may need to restart the kernel to use updated packages.


## BONUS — Step 1.2: Verify RabbitMQ Installation

Check that the RabbitMQ Python client is available in this notebook.

In [3]:
# BONUS — STEP 1.2: VERIFY RABBITMQ INSTALLATION

import pika
from dotenv import load_dotenv

print("RabbitMQ library installed successfully!")
print("Pika version:", pika.__version__)
print("Python executable:", __import__("sys").executable)

RabbitMQ library installed successfully!
Pika version: 1.4.4
Python executable: c:\Users\maddu\AppData\Local\Programs\Python\Python312\python.exe


## BONUS — Step 1.3: Verify Existing FreshCart Pipeline

Check whether the FreshCart landing, Snowflake loading, and SQL
execution functions are available in the current notebook session.

In [4]:
# BONUS — STEP 1.3: VERIFY FRESHCART PIPELINE FUNCTIONS

from pathlib import Path

print("========== FRESHCART PIPELINE CHECK ==========")

print("1. Current working directory:")
print(Path.cwd())

print("\n2. Existing functions:")
print("write_landing:", callable(globals().get("write_landing")))
print("load_file:", callable(globals().get("load_file")))
print("run_sql:", callable(globals().get("run_sql")))

print("\n3. Snowflake connection:")
print("conn available:", "conn" in globals())

print("\n4. Landing directory:")
print("LANDING_DIR:", globals().get("LANDING_DIR"))

print("\n========== CHECK COMPLETED ==========")

========== FRESHCART PIPELINE CHECK ==========
1. Current working directory:
c:\Users\maddu\Desktop\CS8037_Data_Engineering\freshcart-pipelines\elt

2. Existing functions:
write_landing: False
load_file: False
run_sql: False

3. Snowflake connection:
conn available: False

4. Landing directory:
LANDING_DIR: None

========== CHECK COMPLETED ==========


## BONUS — Step 1.4: Final FreshCart Environment Verification

In [21]:
# BONUS — STEP 1.4: FINAL ENVIRONMENT VERIFICATION

print("========== FINAL ENVIRONMENT CHECK ==========")

print("write_landing:", callable(globals().get("write_landing")))
print("load_file:", callable(globals().get("load_file")))
print("run_sql:", callable(globals().get("run_sql")))
print("query:", callable(globals().get("query")))
print("Snowflake connection:", globals().get("conn") is not None)
print("Landing directory:", globals().get("LANDING_DIR"))

print("=============================================")

========== FINAL ENVIRONMENT CHECK ==========
write_landing: True
load_file: True
run_sql: True
query: True
Snowflake connection: True
Landing directory: ./freshcart_landing


## BONUS — Step 2: Configure RabbitMQ Consumer Queue

Create a durable RabbitMQ queue and bind it to the existing
rapido-dispatch topic exchange.

The queue receives FreshCart order events using the routing key
freshcart.orders.created.

In [22]:
# BONUS — STEP 2: CONFIGURE RABBITMQ CONSUMER QUEUE

import os
from pathlib import Path
from dotenv import load_dotenv
import pika

# Load .env from the project root (not the elt folder)
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

RABBITMQ_URL = os.getenv("RABBITMQ_URL")

if not RABBITMQ_URL:
    raise RuntimeError("RABBITMQ_URL not found in project .env file")

EXCHANGE = "rapido-dispatch"
QUEUE = "freshcart-lakehouse-bonus"
ROUTING_KEY = "freshcart.orders.*"

# Connect to RabbitMQ
connection = pika.BlockingConnection(
    pika.URLParameters(RABBITMQ_URL)
)

channel = connection.channel()

# Declare the durable topic exchange
channel.exchange_declare(
    exchange=EXCHANGE,
    exchange_type="topic",
    durable=True
)

# Create a durable queue
queue_info = channel.queue_declare(
    queue=QUEUE,
    durable=True
)

# Bind the queue to the exchange
channel.queue_bind(
    exchange=EXCHANGE,
    queue=QUEUE,
    routing_key=ROUTING_KEY
)

print("RabbitMQ connection successful!")
print("Exchange:", EXCHANGE)
print("Queue:", QUEUE)
print("Binding:", ROUTING_KEY)
print("Messages already waiting:", queue_info.method.message_count)

connection.close()

print("RabbitMQ queue setup completed.")

RabbitMQ connection successful!
Exchange: rapido-dispatch
Queue: freshcart-lakehouse-bonus
Binding: freshcart.orders.*
Messages already waiting: 0
RabbitMQ queue setup completed.


## BONUS — Step 3: Publish FreshCart Orders to RabbitMQ

Publish six sample FreshCart order events through the RabbitMQ
topic exchange using the routing key freshcart.orders.created.

Each message represents a website order in the FreshCart data format.

In [ ]:
# BONUS — STEP 3: PUBLISH FRESHCART ORDERS TO RABBITMQ

import os
import json
from pathlib import Path
from dotenv import load_dotenv
import pika

# Load RabbitMQ connection details from the project root
load_dotenv(dotenv_path=Path.cwd().parent / ".env")
RABBITMQ_URL = os.getenv("RABBITMQ_URL")
if not RABBITMQ_URL:
    raise RuntimeError("RABBITMQ_URL not found in project .env file")

EXCHANGE = "rapido-dispatch"
ROUTING_KEY = "freshcart.orders.created"

# Six sample website orders in the FreshCart flat JSON format
sample_orders = [
    {"order_id": "BONUS-1001", "cust_email": "priya@gmail.com", "cust_name": "Priya R", "order_date": "2026-11-14", "amount": "1250.00", "currency": "INR", "pincode": "600041", "updated_at": "2026-11-14 10:00:00"},
    {"order_id": "BONUS-1002", "cust_email": "arjun@yahoo.in", "cust_name": "Arjun K", "order_date": "2026-11-14", "amount": "850.00", "currency": "INR", "pincode": "560034", "updated_at": "2026-11-14 10:05:00"},
    {"order_id": "BONUS-1003", "cust_email": "meera@outlook.com", "cust_name": "Meera S", "order_date": "2026-11-14", "amount": "999.00", "currency": "INR", "pincode": "400050", "updated_at": "2026-11-14 10:10:00"},
    {"order_id": "BONUS-1004", "cust_email": "rahul@gmail.com", "cust_name": "Rahul M", "order_date": "2026-11-14", "amount": "450.00", "currency": "INR", "pincode": "500081", "updated_at": "2026-11-14 10:15:00"},
    {"order_id": "BONUS-1005", "cust_email": "sneha@gmail.com", "cust_name": "Sneha P", "order_date": "2026-11-14", "amount": "1599.00", "currency": "INR", "pincode": "110001", "updated_at": "2026-11-14 10:20:00"},
    {"order_id": "BONUS-1006", "cust_email": "vikram@gmail.com", "cust_name": "Vikram N", "order_date": "2026-11-14", "amount": "725.00", "currency": "INR", "pincode": "700001", "updated_at": "2026-11-14 10:25:00"},
]

connection = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
try:
    channel = connection.channel()
    channel.exchange_declare(exchange=EXCHANGE, exchange_type="topic", durable=True)
    for order in sample_orders:
        channel.basic_publish(
            exchange=EXCHANGE,
            routing_key=ROUTING_KEY,
            body=json.dumps(order).encode("utf-8"),
            properties=pika.BasicProperties(delivery_mode=2, content_type="application/json"),
            mandatory=True,
        )
        print(f"Published: {order['order_id']}")
    print(f"Total messages published: {len(sample_orders)}")
finally:
    connection.close()


Published: BONUS-1001
Published: BONUS-1002
Published: BONUS-1003
Published: BONUS-1004
Published: BONUS-1005
Published: BONUS-1006
Total messages published: 6


## BONUS — Step 4: Consume RabbitMQ Messages

Consume FreshCart order events from the RabbitMQ queue for 60 seconds.

Parse the JSON messages, write them into a compressed JSON Lines
landing file using the existing write_landing() function, and
acknowledge messages after successful file creation.

In [23]:
# BONUS — STEP 4: CONSUME RABBITMQ MESSAGES

import os
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
import pika

# Load RabbitMQ credentials from project root
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

RABBITMQ_URL = os.getenv("RABBITMQ_URL")

if not RABBITMQ_URL:
    raise RuntimeError("RABBITMQ_URL not found.")

# Verify existing FreshCart function
if not callable(globals().get("write_landing")):
    raise RuntimeError("write_landing() is not available.")

EXCHANGE = "rapido-dispatch"
QUEUE = "freshcart-lakehouse-bonus"

# Unique ID for this landing batch
bonus_run_id = datetime.now(timezone.utc).strftime(
    "BONUS_%Y%m%d_%H%M%S"
)

connection = pika.BlockingConnection(
    pika.URLParameters(RABBITMQ_URL)
)

channel = connection.channel()

channel.queue_declare(
    queue=QUEUE,
    durable=True
)

channel.basic_qos(prefetch_count=20)

orders = []
delivery_tags = []

print("Starting RabbitMQ consumer...")
print("Collecting messages for 60 seconds...")

start_time = time.monotonic()

try:
    while time.monotonic() - start_time < 60:

        method, properties, body = channel.basic_get(
            queue=QUEUE,
            auto_ack=False
        )

        if method is None:
            time.sleep(0.5)
            continue

        try:
            order = json.loads(body.decode("utf-8"))

            # Validate required website order fields
            required_fields = [
                "order_id", "cust_email", "cust_name",
                "order_date", "amount", "currency",
                "pincode", "updated_at"
            ]

            missing = [
                field for field in required_fields
                if field not in order
            ]

            if missing:
                raise ValueError(
                    f"Missing order fields: {missing}"
                )

            orders.append(order)
            delivery_tags.append(method.delivery_tag)

            print(
                f"Received: {order['order_id']}"
            )

        except Exception as error:
            print("Invalid message:", error)
            channel.basic_nack(
                delivery_tag=method.delivery_tag,
                requeue=False
            )

    print("\nCollection period completed.")

    if not orders:
        raise RuntimeError(
            "No valid orders received. Landing file not created."
        )

    # Save records using the existing FreshCart function
    landing_file = write_landing(
        "web",
        orders,
        bonus_run_id
    )

    # Acknowledge only after successful landing-file creation
    for tag in delivery_tags:
        channel.basic_ack(delivery_tag=tag)

    print("\n========== CONSUMER SUMMARY ==========")
    print("Messages received:", len(orders))
    print("Messages acknowledged:", len(delivery_tags))
    print("Run ID:", bonus_run_id)
    print("Landing file:", landing_file)
    print("File exists:", Path(landing_file).exists())
    print("======================================")

finally:
    connection.close()

Starting RabbitMQ consumer...
Received: ORD-STREAM-0001
Received: ORD-STREAM-0002
Received: ORD-STREAM-0003
Received: ORD-STREAM-0004
Received: ORD-STREAM-0005
Received: ORD-STREAM-0006

Collection period completed.
   web: 6 records -> freshcart_landing\web\web__BONUS_20261107_103348.json.gz

========== CONSUMER SUMMARY ==========
Messages received: 6
Messages acknowledged: 6
Run ID: BONUS_20261107_103348
Landing file: freshcart_landing\web\web__BONUS_20261107_103348.json.gz
File exists: True


## BONUS — Step 5: Load RabbitMQ Landing File into Snowflake Bronze

Upload the compressed RabbitMQ landing file into the Snowflake
internal stage and load its six order records into bronze.orders_raw.

In [24]:
# BONUS — STEP 5: LOAD RABBITMQ FILE INTO BRONZE

from pathlib import Path

# Landing file created by the RabbitMQ consumer
bonus_file = Path(
    "freshcart_landing/web/web__BONUS_20261107_103348.json.gz"
)

# Confirm the file exists before loading
if not bonus_file.exists():
    raise FileNotFoundError(
        f"Landing file not found: {bonus_file.resolve()}"
    )

# Record the Bronze row count before loading
bronze_before = query(
    "SELECT COUNT(*) FROM bronze.orders_raw"
)[0][0]

print("Bronze rows before loading:", bronze_before)

# Use a unique stage folder for this batch
bonus_stage_dir = "orders_web/BONUS_20261107_103348"

# Load the RabbitMQ orders into Bronze
load_file(
    str(bonus_file),
    "bronze.orders_raw",
    bonus_stage_dir
)

# Record the Bronze row count after loading
bronze_after = query(
    "SELECT COUNT(*) FROM bronze.orders_raw"
)[0][0]

print("\n========== BRONZE LOAD SUMMARY ==========")
print("Bronze rows before:", bronze_before)
print("Bronze rows after:", bronze_after)
print("Rows added:", bronze_after - bronze_before)
print("Expected rows:", 6)
print("=========================================")

assert bronze_after - bronze_before == 6, (
    "Expected exactly six new Bronze records."
)

print("SUCCESS: Six RabbitMQ orders loaded into Bronze.")

Bronze rows before loading: 16
   6 rows -> bronze.orders_raw

========== BRONZE LOAD SUMMARY ==========
Bronze rows before: 16
Bronze rows after: 22
Rows added: 6
Expected rows: 6
SUCCESS: Six RabbitMQ orders loaded into Bronze.


## BONUS — Step 6.1: Initialize Snowflake Run ID

Set the Snowflake session variable RUN_ID to the RabbitMQ batch ID.
The cleaning SQL uses this variable to identify the current batch.

In [26]:
# BONUS — STEP 6.1: SET SNOWFLAKE RUN_ID

# Use the same batch ID used when creating the landing file
print("Bonus batch ID:", bonus_run_id)

# Initialize the Snowflake session variable
cur = conn.cursor()

try:
    cur.execute(
        f"SET RUN_ID = '{bonus_run_id}'"
    )

    print("Snowflake RUN_ID initialized successfully.")

finally:
    cur.close()

Bonus batch ID: BONUS_20261107_103348
Snowflake RUN_ID initialized successfully.


## BONUS — Step 6.2: Inspect Silver Table Columns

Inspect the actual columns in silver.orders_clean so that the
verification query uses the correct Snowflake column names.

In [ ]:
# BONUS — STEP 6.2: INSPECT SILVER TABLE COLUMNS

cur = conn.cursor()

try:
    cur.execute("DESCRIBE TABLE silver.orders_clean")

    columns = cur.fetchall()

    print("========== SILVER TABLE COLUMNS ==========")

    for column in columns:
        print(column[0], "| Type:", column[1])

    print("===========================================")

finally:
    cur.close()

========== SILVER TABLE COLUMNS ==========
INGEST_SEQ | Type: NUMBER(38,0)
SOURCE | Type: VARCHAR(16777216)
ORDER_ID | Type: VARCHAR(16777216)
EMAIL | Type: VARCHAR(16777216)
NAME | Type: VARCHAR(16777216)
ORDER_DATE | Type: DATE
AMOUNT | Type: NUMBER(12,2)
CURRENCY | Type: VARCHAR(16777216)
PINCODE | Type: VARCHAR(16777216)
UPDATED_AT | Type: TIMESTAMP_NTZ(9)
AMOUNT_MISSING | Type: BOOLEAN


## BONUS — Step 6.3: Verify RabbitMQ Orders in Silver

Display the six streamed orders after the FreshCart cleaning
and validation transformation.

In [ ]:
# BONUS — STEP 6.3: VERIFY SILVER ORDERS

show("""
SELECT
    ORDER_ID,
    EMAIL,
    NAME,
    ORDER_DATE,
    AMOUNT,
    CURRENCY,
    PINCODE
FROM silver.orders_clean
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
ORDER BY ORDER_ID
""")

,ORDER_ID,EMAIL,NAME,ORDER_DATE,AMOUNT,CURRENCY,PINCODE
0,ORD-STREAM-0001,meera@outlook.com,Meera S,2026-09-03,500.00,INR,400050
1,ORD-STREAM-0002,meera@outlook.com,Meera S,2026-09-03,500.00,INR,400050
2,ORD-STREAM-0003,meera@outlook.com,Meera S,2026-09-03,500.00,INR,400050
3,ORD-STREAM-0004,meera@outlook.com,Meera S,2026-09-03,500.00,INR,400050
4,ORD-STREAM-0005,meera@outlook.com,Meera S,2026-09-03,500.00,INR,400050
5,ORD-STREAM-0006,meera@outlook.com,Meera S,2026-09-03,500.00,INR,400050


## BONUS — Step 7: Deduplicate Silver Orders

Execute the existing SQL_20_ORDERS_DEDUP transformation.
Verify that all six streamed orders appear once in the deduplicated
Silver table.

In [30]:
# BONUS — STEP 7: DEDUPLICATE ORDERS

run_sql(SQL_20_ORDERS_DEDUP)

print("Deduplication completed successfully.")

show("""
SELECT
    ORDER_ID,
    EMAIL,
    NAME,
    ORDER_DATE,
    AMOUNT,
    CURRENCY,
    PINCODE
FROM silver.orders_dedup
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
ORDER BY ORDER_ID
""")

print("\nStreamed order count in deduplicated Silver:")

show("""
SELECT COUNT(*) AS STREAMED_ORDER_COUNT
FROM silver.orders_dedup
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
""")

Deduplication completed successfully.

Streamed order count in deduplicated Silver:


,STREAMED_ORDER_COUNT
0,6


## BONUS — Step 8: Prepare Reference and Customer Enrichment

Execute the existing FreshCart transformations for customer SCD Type 2,
daily foreign exchange rates, and pincode geography.

In [31]:
# BONUS — STEP 8: CUSTOMER AND REFERENCE ENRICHMENT

print("Starting FreshCart enrichment transformations...")

run_sql(SQL_30_CUSTOMERS_SCD2)
print("1. Customer SCD2 transformation completed.")

run_sql(SQL_31_FX_DAILY)
print("2. Daily FX transformation completed.")

run_sql(SQL_32_PINCODE_GEO)
print("3. Pincode geography transformation completed.")

print("\nAll enrichment transformations completed.")

Starting FreshCart enrichment transformations...
1. Customer SCD2 transformation completed.
2. Daily FX transformation completed.
3. Pincode geography transformation completed.

All enrichment transformations completed.


## BONUS — Step 9: Publish RabbitMQ Orders into Gold

Execute the existing SQL_40_GOLD transformation to enrich the
deduplicated orders and publish them into gold.fct_orders.

Verify that all six streamed order IDs are present in Gold.

In [32]:
# BONUS — STEP 9: PUBLISH ORDERS INTO GOLD

# Record Gold totals before the transformation
gold_before = query("""
SELECT
    COUNT(*),
    COALESCE(SUM(amount_inr), 0),
    COALESCE(SUM(loyalty_points), 0)
FROM gold.fct_orders
""")[0]

print("Gold totals before:", gold_before)

# Execute the existing Gold MERGE
run_sql(SQL_40_GOLD)

print("\nSQL_40_GOLD executed successfully.")

# Display streamed orders in Gold
show("""
SELECT *
FROM gold.fct_orders
WHERE order_id LIKE 'ORD-STREAM-%'
ORDER BY order_id
""")

# Record Gold totals after the transformation
gold_after = query("""
SELECT
    COUNT(*),
    COALESCE(SUM(amount_inr), 0),
    COALESCE(SUM(loyalty_points), 0)
FROM gold.fct_orders
""")[0]

print("\nGold totals after:", gold_after)

print("\n========== GOLD SUMMARY ==========")
print("Total Gold rows before:", gold_before[0])
print("Total Gold rows after:", gold_after[0])
print("Total Gold revenue after:", gold_after[1])
print("Total Gold loyalty points after:", gold_after[2])
print("==================================")

Gold totals before: (4, Decimal('7785.00'), 161)

SQL_40_GOLD executed successfully.

Gold totals after: (10, Decimal('10785.00'), 251)

========== GOLD SUMMARY ==========
Total Gold rows before: 4
Total Gold rows after: 10
Total Gold revenue after: 10785.00
Total Gold loyalty points after: 251


## BONUS — Step 10: Verify Gold Idempotency

Execute the Gold MERGE transformation a second time using the
same Silver data.

Verify that Gold order count, revenue, and loyalty points remain
unchanged after replay.

In [33]:
# BONUS — STEP 10: GOLD IDEMPOTENCY TEST

# Capture totals before replay
before_replay = query("""
SELECT
    COUNT(*),
    COALESCE(SUM(amount_inr), 0),
    COALESCE(SUM(loyalty_points), 0)
FROM gold.fct_orders
""")[0]

print("Gold totals before replay:", before_replay)

# Replay the same Gold MERGE
run_sql(SQL_40_GOLD)

print("\nGold MERGE executed again successfully.")

# Capture totals after replay
after_replay = query("""
SELECT
    COUNT(*),
    COALESCE(SUM(amount_inr), 0),
    COALESCE(SUM(loyalty_points), 0)
FROM gold.fct_orders
""")[0]

print("Gold totals after replay:", after_replay)

# Verify streamed order IDs are unique in Gold
show("""
SELECT
    COUNT(*) AS TOTAL_STREAMED_ROWS,
    COUNT(DISTINCT ORDER_ID) AS UNIQUE_STREAMED_ORDERS
FROM gold.fct_orders
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
""")

# Verify no Gold totals changed
assert before_replay == after_replay, (
    "Idempotency test failed: Gold totals changed after replay."
)

print("\nSUCCESS: Gold totals did not change after replay.")
print("SUCCESS: Gold MERGE is idempotent.")

Gold totals before replay: (10, Decimal('10785.00'), 251)

Gold MERGE executed again successfully.
Gold totals after replay: (10, Decimal('10785.00'), 251)

SUCCESS: Gold totals did not change after replay.
SUCCESS: Gold MERGE is idempotent.


## BONUS — Step 11: Final Gold Verification

Verify the number of streamed orders, unique order IDs,
total revenue, and loyalty points in the Gold fact table.

In [34]:
# BONUS — STEP 11: FINAL GOLD VERIFICATION

print("========== FINAL STREAMED ORDER CHECK ==========")

show("""
SELECT
    COUNT(*) AS TOTAL_ROWS,
    COUNT(DISTINCT ORDER_ID) AS UNIQUE_ORDERS,
    SUM(AMOUNT_INR) AS TOTAL_REVENUE,
    SUM(LOYALTY_POINTS) AS TOTAL_POINTS
FROM gold.fct_orders
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
""")

print("\nIndividual streamed orders:")

show("""
SELECT
    ORDER_ID,
    AMOUNT_INR,
    LOYALTY_POINTS,
    ROW_HASH
FROM gold.fct_orders
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
ORDER BY ORDER_ID
""")

print("\n========== FINAL CHECK COMPLETED ==========")

========== FINAL STREAMED ORDER CHECK ==========

Individual streamed orders:

========== FINAL CHECK COMPLETED ==========


## BONUS — Step 11.1: Verify Streamed Orders Directly in Gold

Retrieve the six RabbitMQ orders directly from Snowflake and
verify their unique IDs, revenue, and loyalty points.

In [35]:
# BONUS — STEP 11.1: DIRECT GOLD VERIFICATION

print("========== STREAMED ORDERS IN GOLD ==========")

rows = query("""
SELECT
    ORDER_ID,
    AMOUNT_INR,
    LOYALTY_POINTS
FROM gold.fct_orders
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
ORDER BY ORDER_ID
""")

for row in rows:
    print(row)

print("\n========== GOLD AGGREGATE CHECK ==========")

totals = query("""
SELECT
    COUNT(*),
    COUNT(DISTINCT ORDER_ID),
    SUM(AMOUNT_INR),
    SUM(LOYALTY_POINTS)
FROM gold.fct_orders
WHERE ORDER_ID LIKE 'ORD-STREAM-%'
""")[0]

print("Total rows:", totals[0])
print("Unique orders:", totals[1])
print("Total revenue:", totals[2])
print("Total loyalty points:", totals[3])

assert totals[0] == 6
assert totals[1] == 6
assert float(totals[2]) == 3000.0
assert totals[3] == 90

print("\nSUCCESS: All six streamed orders verified in Gold.")

========== STREAMED ORDERS IN GOLD ==========
('ORD-STREAM-0001', Decimal('500.00'), 15)
('ORD-STREAM-0002', Decimal('500.00'), 15)
('ORD-STREAM-0003', Decimal('500.00'), 15)
('ORD-STREAM-0004', Decimal('500.00'), 15)
('ORD-STREAM-0005', Decimal('500.00'), 15)
('ORD-STREAM-0006', Decimal('500.00'), 15)

========== GOLD AGGREGATE CHECK ==========
Total rows: 6
Unique orders: 6
Total revenue: 3000.00
Total loyalty points: 90

SUCCESS: All six streamed orders verified in Gold.
